# Causal BC on HumanoidMaze Large


In [1]:
import random
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
hidden/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 2000
hidden_dims = {'C'}

In [4]:
# for eval: corrupted W, C hidden
eval_env = HumanoidMazePCH(env_id='humanoidmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=False, success_radius=15.0)

In [5]:
# load model
MODEL_PATH = 'hidden/cbc_humlarge.pt'
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

action_bounds = (ckpt['action_bounds_low'], ckpt['action_bounds_high'])

causal_bc_model = ContinuousPolicyNN(
    input_dim=ckpt['input_dim'],
    action_dim=ckpt['num_actions'],
    hidden_dim=ckpt['hidden_dim'],
    num_blocks=ckpt['num_blocks'],
    dropout=ckpt['dropout'],
    layernorm=ckpt['layernorm'],
    final_tanh=ckpt['final_tanh'],
    action_bounds=action_bounds,
).to(device)

causal_bc_model.load_state_dict(ckpt['state_dict'])
causal_bc_model.eval()

slots = ckpt['slots']
Z_trim = ckpt['Z_trim']
dims = ckpt['dims']
lookback = ckpt['lookback']

cbc_policy = shared_policy_fn_long_horizon(causal_bc_model, slots, Z_trim, continuous=True, device=device)
cbc_policies = make_shared_policy_dict(cbc_policy)

## Evaluation

In [6]:
num_eval_eps = 1000
cbc_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=cbc_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=1
)

len(cbc_returns)

Starting episode 1/1000...


  Episode 1 ended at step 1351 (terminated: True, truncated: False).
Starting episode 2/1000...


  Episode 2 ended at step 819 (terminated: True, truncated: False).
Starting episode 3/1000...


  Episode 3 ended at step 2000 (terminated: False, truncated: True).
Starting episode 4/1000...


  Episode 4 ended at step 2000 (terminated: False, truncated: True).
Starting episode 5/1000...


...

  Episode 21 ended at step 2000 (terminated: False, truncated: True).
Starting episode 22/1000...


  Episode 22 ended at step 2000 (terminated: False, truncated: True).
Starting episode 23/1000...


  Episode 23 ended at step 2000 (terminated: False, truncated: True).
Starting episode 24/1000...


  Episode 24 ended at step 2000 (terminated: False, truncated: True).
Starting episode 25/1000...


  Episode 25 ended at step 2000 (terminated: False, truncated: True).
Starting episode 26/1000...


In [ ]:
cbc_episode_rewards = defaultdict(float)
for rec in cbc_returns:
    ep = rec['episode']
    cbc_episode_rewards[ep] += float(rec['reward'])

cbc_rewards = [cbc_episode_rewards[e] for e in range(num_eval_eps)]
sum(cbc_rewards) / num_eval_eps

In [ ]:
mean_reward = np.mean(cbc_rewards)
std_reward = np.std(cbc_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] ± Std[Y] = {mean_reward:.4f} ± {std_reward:.4f}")

In [ ]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in cbc_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

In [ ]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")